In [1]:
from the_well.benchmark.models import UNetClassic, UNetConvNext, FNO, TFNO
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import pandas as pd
import numpy as np
import fsspec
import pickle
import torch
import yaml
import sys

from pathlib import Path as _Path
PROJECT_ROOT = str(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "pyproject.toml").exists()))  # repo root, works on laptop and cluster
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.utils.eval import build_stats_df, build_psd_df
from waveletflux.modules.neural_ops import NeuralOperatorModule
from waveletflux.utils.wrappers import TFNOMPSSafeWrapper
from waveletflux.utils.rollout import rollout_normalized

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Helpers

In [2]:
def label_to_dict(label: torch.Tensor | None, scalar_names: list[str]) -> dict[str, float]:
    """Convert a 1-D label tensor to a {name: float} dict.

    If scalar_names is shorter than the tensor (or empty), unnamed entries are
    stored as 'cond_0', 'cond_1', … so no information is ever silently dropped.
    """
    if label is None:
        return {}
    values = label.detach().cpu().float().tolist()
    names = list(scalar_names) + [
        f"cond_{i}" for i in range(len(scalar_names), len(values))
    ]
    return {n: v for n, v in zip(names, values)}

def build_traj_index(
    n_traj_per_file:   list[int],
    steps_per_file:    list[int],
    traj_per_file_cap: int,
) -> tuple[list[int], list[int]]:
    """Return (dataset_indices, n_steps) for the selected trajectories.

    Parameters
    ----------
    n_traj_per_file   : list of trajectory counts per file  (from dataset)
    steps_per_file    : list of step counts per file        (from metadata)
    traj_per_file_cap : how many trajectories to take from each file;
                        -1 means take all.

    Returns
    -------
    traj_indices   : flat list of integer indices into the dataset
    n_steps_list   : corresponding number of steps for each selected trajectory
    """
    traj_indices = []
    n_steps_list = []
    offset = 0
    for n_traj, n_steps in zip(n_traj_per_file, steps_per_file):
        cap = n_traj if traj_per_file_cap == -1 else min(traj_per_file_cap, n_traj)
        for local_i in range(cap):
            traj_indices.append(offset + local_i)
            n_steps_list.append(n_steps)
        offset += n_traj
    return traj_indices, n_steps_list

def build_labels_df(
    labels_dict: dict[int, dict[str, float]],
    traj_indices: list[int],
) -> pd.DataFrame:
    """Build a tidy DataFrame with one row per trajectory and one column per
    scalar condition.  ``traj`` matches the key used in stats / psd DataFrames
    so it can be directly merged on that column.

    Example output for the Reyleigh-Bénard dataset:
        traj  Rayleigh  Prandtl
           0   23.0259      1.0
           1    9.2103      0.5
    """
    rows = [{"traj": idx, **labels_dict.get(idx, {})} for idx in traj_indices]
    return pd.DataFrame(rows)

# Load Data

In [14]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

#well_dataset_name = 'turbulent_radiative_layer_2D'
well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'
#well_dataset_name = 'active_matter'

if well_dataset_name == 'turbulent_radiative_layer_2D':
    transform = [] #['normalize_turbulent_radiative_scalars']
elif well_dataset_name == 'rayleigh_benard':
    transform = ['log_scalars']
elif well_dataset_name == 'active_matter':
    transform = [] #['normalize_active_matter_scalars']
else:    transform = [None]

stats_path = f"{well_base_path}/{well_dataset_name}/stats.yaml"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = transform,   
    augment = [],
)

scalar_names = list(getattr(dataset.metadata, "constant_scalar_names", []) or [])

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Trajectory sub-sampling
N_TRAJ_PER_FILE = 1   # N = first N per file
#N_TRAJ_PER_FILE = -1 # -1 = all trajectories

# Load Baseline Models

In [15]:
#unet_classic  = UNetClassic.from_pretrained(f"polymathic-ai/UNetClassic-{well_dataset_name}")
#unet_convnext = UNetConvNext.from_pretrained(f"polymathic-ai/UNetConvNext-{well_dataset_name}")
#fno = FNO.from_pretrained(f"polymathic-ai/FNO-{well_dataset_name}")
#tfno = TFNO.from_pretrained(f"polymathic-ai/TFNO-{well_dataset_name}")
fno           = NeuralOperatorModule.from_pretrained(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/FNO_{well_dataset_name}.pt")
tfno          = NeuralOperatorModule.from_pretrained(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/TFNO_{well_dataset_name}.pt")

baselines = {
    #'UNetClassic'   : unet_classic.to(device),
    #'UNetConvNext'  : unet_convnext.to(device),
    'FNO'           : fno.to(device),
    'TFNO'          : TFNOMPSSafeWrapper(tfno.to(device))
}

In [17]:
n_params = sum(p.numel() for p in fno.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")
# fno 19,011,716
#tfno 7,922,524

Parameters: 19,011,716


In [ ]:
# ── Build trajectory index ────────────────────────────────────────────────
traj_indices, n_steps_per_traj = build_traj_index(
    n_traj_per_file   = dataset.n_trajectories_per_file,
    steps_per_file    = dataset.metadata.n_steps_per_trajectory,
    traj_per_file_cap = N_TRAJ_PER_FILE,
)
N_TRAJ        = len(traj_indices)
N_STEPS_INPUT = 4
print(f"Evaluating {N_TRAJ} trajectories ({N_TRAJ_PER_FILE} TRAJ for each of the {len(dataset.n_trajectories_per_file)} files)")

# ── Inference loop ────────────────────────────────────────────────────────
baseline_results = {name: [] for name in baselines}
truth_list       = []
labels_dict      = {}

with torch.no_grad():
    for i, (traj_idx, n_steps) in enumerate(zip(traj_indices, n_steps_per_traj)):
        rollout_steps = n_steps - N_STEPS_INPUT

        print(f"\n{'━'*60}")
        print(f"  Position {i}  (dataset_idx={traj_idx})")
        print(f"{'━'*60}")

        item = dataset[traj_idx]
        x_full, constants = get_well_inputs(item, device=device)
        truth_list.append(x_full.cpu())

        labels_dict[i] = label_to_dict(constants, scalar_names)  # keyed by i, not traj_idx

        for model_name, model in baselines.items():
            print(f"    Running rollout for {model_name}...")
            model.eval()
            _, _, y_pred_phys, _ = rollout_normalized(
                model, x_full, normalizer,
                n_steps_input = N_STEPS_INPUT,
                rollout_steps = rollout_steps,
                device        = device,
            )
            baseline_results[model_name].append(y_pred_phys)

# Stack to (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
for name in baselines:
    baseline_results[name] = torch.stack(baseline_results[name], dim=0)

truth_traj = torch.stack(truth_list, dim=0) # (N_TRAJ, T_total, H, W, C)

# Save baselines to binary

In [ ]:
# save baseline trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(baseline_results[model_name], f)

In [ ]:
# double check shapes
for model_name in baselines.keys():
    print(baseline_results[model_name].shape)

# Save Ground-truth to binary

In [ ]:
# save true trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(truth_traj, f)

In [ ]:
truth_traj.shape

# Load other baselines (WNO and WDNO)

In [ ]:
model_name  = 'WNO'

# 1-scale model(s)
#model_path = f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results/{model_name}_1scale_db2_{well_dataset_name}_traj.pkl'
# 3-scales model(s)
model_path = f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results/{model_name}_3scales_db2_{well_dataset_name}_traj.pkl'
model = pickle.load(open(model_path, "rb"))
  
truth_path = f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results/ground_truth_{well_dataset_name}_traj.pkl'
truth_traj = pickle.load(open(truth_path, "rb"))
   
baselines = {model_name: model}
baseline_results = {model_name: model}

# ── Build trajectory index ────────────────────────────────────────────────
traj_indices, n_steps_per_traj = build_traj_index(
    n_traj_per_file   = dataset.n_trajectories_per_file,
    steps_per_file    = dataset.metadata.n_steps_per_trajectory,
    traj_per_file_cap = N_TRAJ_PER_FILE,
)
N_TRAJ = len(traj_indices)

labels_dict = {}
for i, traj_idx in enumerate(traj_indices):
    item = dataset[traj_idx]
    _, constants = get_well_inputs(item, device=device)
    labels_dict[i] = label_to_dict(constants, scalar_names)
   
N_STEPS_INPUT = 4

# Compute stats and PSD

In [ ]:
meta_fields = dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]
labels_df = build_labels_df(labels_dict, list(range(N_TRAJ)))

for model_name in baselines.keys():
    print(f"Computing stats for {model_name} ...")
    save_dir    = Path(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/metrics")
    predictions = {model_name: baseline_results[model_name]}

    stats_df = build_stats_df(
        truth       = truth_traj,
        predictions = predictions,
        field_names = field_names,
        Ti          = N_STEPS_INPUT,
        lead_times  = None,
        #stride      = 1,
    )
    if not labels_df.empty:
        stats_df = stats_df.merge(labels_df, on="traj", how="left")
    stats_df.to_csv(save_dir / f"{model_name}_{well_dataset_name}_stats.csv", index=False)

    psd_df = build_psd_df(
        truth         = truth_traj,
        predictions   = predictions,
        field_names   = field_names,
        Ti            = N_STEPS_INPUT,
        lead_times    = None,
        #stride        = 1,
        min_bin_count = 3,
    )
    if not labels_df.empty:
        psd_df = psd_df.merge(labels_df, on="traj", how="left")
    psd_df.to_csv(save_dir / f"{model_name}_{well_dataset_name}_psd.csv", index=False)